# LightGBM text + gene-NMF 독립 모델 — Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cancer-classification-ai/onco-ai/blob/feat/rwr-stacking/notebooks/12_lgbm_text_nmf_colab.ipynb)

RWR 없이 첫 번째 독립 LightGBM 모델과 ablation을 실행한다.

고정 조건:

- CV: canonical `fold_group5`, 5-fold
- class order: 현재 26개 클래스 알파벳순
- model seed: 42
- 기본 모델: `domain + rollup16 + sigtok + ptok + lnmf(64)`
- exact token: 최종 후보가 아닌 coverage 대조군이므로 제외
- 모델 비교: **Pair rule 적용 전 raw OOF**
- Pair rule: 이 노트북에서 호출하지 않음
- 하이퍼파라미터 튜닝 없음

실행 순서는 기본 → 단일 블록 제거 → NMF 32/128 → TF-IDF top-K 500/2000이다.
NMF 64와 TF-IDF 1000은 기본 실험 결과를 재사용한다.

## 0. 설정

`DATA_SOURCE`는 `kagglehub` 또는 `drive`를 선택한다. v002 raw OOF가 있으면 마지막
평가 셀을 위해 `V002_OOF`에 경로를 입력한다.

In [ ]:
from __future__ import annotations

import json
import os
import platform
import shutil
import subprocess
import sys
import time
import traceback
from datetime import datetime, timezone
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/cancer-classification-ai/onco-ai.git"
PROJECT_BRANCH = "feat/rwr-stacking"

# 데이터: 기존 팀 Kaggle Dataset 또는 Google Drive 폴더
DATA_SOURCE = "kagglehub"  # "kagglehub" / "drive"
KAGGLE_DATASET = "hyunwoo11/onco-data-hack"
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/onco-data-hack")
MOUNT_GOOGLE_DRIVE = True

SEED = 42
N_SPLITS = 5
CV = "sgkf"  # train_folds.parquet의 fold_group5
TAG = "stack_lgbm_v1"
DEVICE = "cpu"  # Colab 기본 LightGBM wheel은 CPU 실행
THREADS = None   # None이면 전체 CPU 코어
RESUME_COMPLETED = True

WORK_ROOT = Path("/content") if IN_COLAB else Path.cwd()
REPO_DIR = WORK_ROOT / "onco-ai-rwr"
if Path.cwd().name == "onco-ai" and (Path.cwd() / "scripts").is_dir():
    REPO_DIR = Path.cwd()

OUTPUT_ROOT = (
    Path("/content/drive/MyDrive/onco_lgbm_text_nmf")
    if IN_COLAB
    else REPO_DIR / "artifacts/lgbm_text_nmf_colab"
)

# 선택 사항: Pair rule이 적용되지 않은 v002 raw OOF 경로.
# 내부 형식(ID, p_<CLASS>, y_true)과 팀 공유 형식(sample_id, prob_class_N,
# true_label)을 모두 지원한다. 비워 두면 50:50 평가는 건너뛴다.
V002_OOF = ""

print("environment :", "Colab" if IN_COLAB else "local")
print("repo        :", REPO_DIR)
print("output      :", OUTPUT_ROOT)
print("CV/seed     :", CV, SEED)
print("Pair rule   : disabled")

## 1. Drive, 저장소, 패키지 준비

새 Colab 런타임에서는 지정 branch를 clone한다. 이미 clone된 저장소는 자동 pull하지
않으며 현재 commit을 기록한다.

In [ ]:
if IN_COLAB and MOUNT_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)

if not (REPO_DIR / ".git").is_dir():
    subprocess.run(
        [
            "git", "clone", "--branch", PROJECT_BRANCH, "--single-branch",
            REPO_URL, str(REPO_DIR),
        ],
        check=True,
    )

# 예측에 직접 영향을 주는 두 패키지는 팀 고정 버전을 사용한다.
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "lightgbm==4.7.0", "scikit-learn==1.9.0",
        "numpy", "pandas", "scipy", "pyarrow", "joblib", "PyYAML",
        "kagglehub",
    ],
    check=True,
)

os.chdir(REPO_DIR)
commit_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
branch_name = subprocess.check_output(
    ["git", "branch", "--show-current"], text=True
).strip()
print("branch:", branch_name)
print("commit:", commit_sha)

driver_text = (REPO_DIR / "scripts/train_gbdt.py").read_text(encoding="utf-8")
required_code = ("lgbm_text_nmf", "--tfidf-topk", "EXPLICIT_ONLY_CONFIGS")
missing_code = [token for token in required_code if token not in driver_text]
if missing_code:
    raise RuntimeError(
        f"현재 branch에 LightGBM 실험 코드가 없습니다: {missing_code}. "
        "feat/rwr-stacking 최신 commit을 확인하세요."
    )

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

## 2. 데이터 연결

지원 layout은 `<root>/raw + <root>/process`, `<root>/data/raw + data/process`,
또는 root에 CSV가 직접 있는 형식이다. process cache는 저장소 작업 폴더에 symlink한다.

In [ ]:
def valid_raw(path: Path) -> bool:
    return all((path / name).is_file() for name in ("train.csv", "test.csv"))


def resolve_layout(root: Path) -> tuple[Path, Path | None]:
    candidates = [
        (root / "raw", root / "process"),
        (root / "data/raw", root / "data/process"),
        (root, root / "process"),
    ]
    for raw, process in candidates:
        if valid_raw(raw):
            return raw.resolve(), process.resolve() if process.is_dir() else None
    discovered = sorted(Path("/content").glob("**/train.csv")) if IN_COLAB else []
    for train_path in discovered:
        raw = train_path.parent
        if valid_raw(raw):
            process = raw.parent / "process"
            return raw.resolve(), process.resolve() if process.is_dir() else None
    raise FileNotFoundError(f"train.csv/test.csv를 찾지 못했습니다: {root}")


if DATA_SOURCE == "kagglehub":
    if IN_COLAB and not os.environ.get("KAGGLE_API_TOKEN"):
        try:
            from google.colab import userdata

            token = userdata.get("KAGGLE_API_TOKEN")
            if token:
                os.environ["KAGGLE_API_TOKEN"] = token
        except Exception:
            pass
    import kagglehub

    try:
        data_root = Path(
            kagglehub.dataset_download(
                KAGGLE_DATASET,
                output_dir=str(WORK_ROOT / "onco-data-hack"),
            )
        )
    except Exception as exc:
        raise RuntimeError(
            "Kaggle Dataset 다운로드 실패. Colab Secrets의 KAGGLE_API_TOKEN과 "
            "Dataset 접근 권한을 확인하세요."
        ) from exc
elif DATA_SOURCE == "drive":
    data_root = DRIVE_DATA_DIR
else:
    raise ValueError("DATA_SOURCE는 'kagglehub' 또는 'drive'여야 합니다.")

RAW_SOURCE, PROCESS_SOURCE = resolve_layout(data_root)
RAW_DIR = REPO_DIR / "data/raw"
PROCESS_DIR = REPO_DIR / "data/process"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESS_DIR.mkdir(parents=True, exist_ok=True)

for name in ("train.csv", "test.csv", "sample_submission.csv"):
    source = RAW_SOURCE / name
    if not source.is_file():
        raise FileNotFoundError(source)
    target = RAW_DIR / name
    if not target.exists():
        target.symlink_to(source)

if PROCESS_SOURCE is not None and PROCESS_SOURCE != PROCESS_DIR.resolve():
    for source in PROCESS_SOURCE.iterdir():
        if source.is_file():
            target = PROCESS_DIR / source.name
            if not target.exists():
                target.symlink_to(source)

print("raw source    :", RAW_SOURCE)
print("process source:", PROCESS_SOURCE)
print("cached parquet:", len(list(PROCESS_DIR.glob("*.parquet"))))

## 3. 필요한 cache만 생성

이미 있는 파일은 재사용한다. NMF와 TF-IDF는 저장 cache가 아니라 각 outer fold의
train 부분에서만 fit된다.

In [ ]:
FEATURE_COMMANDS = {
    "domain_features.parquet": ["domain"],
    "sample_mutation_features_rollup.parquet": ["sample", "--include-cell-rollup"],
    "signature_mutation_tokens.parquet": ["sigtokens"],
    "parsed_mutation_tokens.parquet": ["parsed-tokens"],
    "mutation_encoded.parquet": ["enc3"],
}

for split in ("train", "test"):
    for suffix, command in FEATURE_COMMANDS.items():
        output = PROCESS_DIR / f"{split}_{suffix}"
        if output.is_file():
            continue
        cmd = [
            sys.executable, "scripts/make_features.py", command[0],
            "--split", split,
            "--input", str(RAW_DIR / f"{split}.csv"),
            "--output", str(output),
            *command[1:],
        ]
        print("RUN:", " ".join(cmd))
        subprocess.run(cmd, check=True)

fold_path = PROCESS_DIR / "train_folds.parquet"
if not fold_path.is_file():
    subprocess.run(
        [
            sys.executable, "scripts/make_folds.py",
            "--input", str(RAW_DIR / "train.csv"),
            "--out", str(fold_path),
            "--n-splits", str(N_SPLITS),
            "--seed", str(SEED),
        ],
        check=True,
    )

expected = [
    PROCESS_DIR / f"{split}_{suffix}"
    for split in ("train", "test")
    for suffix in FEATURE_COMMANDS
] + [fold_path]
missing = [str(path) for path in expected if not path.is_file()]
if missing:
    raise FileNotFoundError("누락된 cache:\n" + "\n".join(missing))
print(f"cache check OK: {len(expected)} files")

## 4. 고정 조건 검증

학습 전에 fold, class order, config, seed를 assert한다. 조건이 다르면 실행하지 않는다.

In [ ]:
sys.path.insert(0, str(REPO_DIR / "src"))
sys.path.insert(0, str(REPO_DIR / "scripts"))

import numpy as np
import pandas as pd
import sklearn
import lightgbm

import train_gbdt as tg
from cancer_hack.class_order import CANONICAL_CLASS_ORDER

tg.RAW_DIR = RAW_DIR
tg.PROC_DIR = PROCESS_DIR
tg.ARTIFACTS = OUTPUT_ROOT / "artifacts"
tg.ARTIFACTS.mkdir(parents=True, exist_ok=True)

labels = pd.read_csv(RAW_DIR / "train.csv", usecols=["ID", "SUBCLASS"], dtype=str)
observed_classes = tuple(sorted(labels["SUBCLASS"].unique()))
assert observed_classes == tuple(CANONICAL_CLASS_ORDER)
assert len(observed_classes) == 26

folds = pd.read_parquet(fold_path)
fold_meta_path = fold_path.with_suffix(".json")
if not fold_meta_path.is_file():
    raise RuntimeError(
        "canonical fold metadata(train_folds.json)가 없습니다. "
        "출처가 불명확한 fold로 OOF를 만들지 않습니다."
    )
with fold_meta_path.open(encoding="utf-8") as handle:
    fold_meta = json.load(handle)
assert fold_meta["seed"] == 42
assert fold_meta["n_splits"] == N_SPLITS
assert "fold_group5" in fold_meta["fold_columns"]
assert "fold_group5" in folds.columns
assert set(folds["fold_group5"].astype(int)) == set(range(N_SPLITS))
assert folds.groupby("group_key")["fold_group5"].nunique().max() == 1
assert folds["ID"].astype(str).tolist() == labels["ID"].astype(str).tolist()

base_blocks = ("domain", "rollup16", "sigtok", "ptok", "lnmf")
assert tg.CONFIGS["lgbm_text_nmf"]["blocks"] == base_blocks
assert tg.CONFIGS["lgbm_text_nmf"]["weight"] == "balanced"
assert "exacttok" not in base_blocks
assert SEED == 42 and CV == "sgkf" and N_SPLITS == 5

print("python       :", platform.python_version())
print("lightgbm     :", lightgbm.__version__)
print("scikit-learn:", sklearn.__version__)
print("classes      :", len(observed_classes), observed_classes[:3], "...", observed_classes[-3:])
print("fold_group5  :", folds["fold_group5"].value_counts().sort_index().to_dict())
print("config       :", base_blocks)

## 5. 실험 목록과 공용 Dataset

총 8개 실험이다. 기본값과 중복되는 NMF 64 및 TF-IDF 1000은 다시 실행하지 않는다.
`--tfidf-topk`만 변경하므로 parsed-token top-K는 항상 1000이다.

In [ ]:
EXPERIMENTS = [
    {"key": "base", "config": "lgbm_text_nmf", "nmf": 64, "tfidf": 1000,
     "desc": "base: domain + rollup16 + sigtok + ptok + lnmf64"},
    {"key": "minus_sigtok", "config": "lgbm_text_nmf_no_sigtok", "nmf": 64, "tfidf": 1000,
     "desc": "single-block ablation: -sigtok"},
    {"key": "minus_ptok", "config": "lgbm_text_nmf_no_ptok", "nmf": 64, "tfidf": 1000,
     "desc": "single-block ablation: -ptok"},
    {"key": "minus_lnmf", "config": "lgbm_text_nmf_no_lnmf", "nmf": 64, "tfidf": 1000,
     "desc": "single-block ablation: -lnmf"},
    {"key": "nmf32", "config": "lgbm_text_nmf", "nmf": 32, "tfidf": 1000,
     "desc": "NMF components 32"},
    {"key": "nmf128", "config": "lgbm_text_nmf", "nmf": 128, "tfidf": 1000,
     "desc": "NMF components 128"},
    {"key": "tfidf500", "config": "lgbm_text_nmf", "nmf": 64, "tfidf": 500,
     "desc": "signature TF-IDF top-K 500; ptok fixed 1000"},
    {"key": "tfidf2000", "config": "lgbm_text_nmf", "nmf": 64, "tfidf": 2000,
     "desc": "signature TF-IDF top-K 2000; ptok fixed 1000"},
]

all_needed = set().union(
    *(set(tg.CONFIGS[exp["config"]]["blocks"]) for exp in EXPERIMENTS)
)
data = tg.Dataset(all_needed, n_splits=N_SPLITS, folds_path=fold_path)
assert tuple(data.classes.tolist()) == observed_classes

STATE_PATH = OUTPUT_ROOT / "suite_state.json"
FAILURE_PATH = OUTPUT_ROOT / "failed_runs.json"


def load_json(path: Path, default):
    if not path.is_file():
        return default
    with path.open(encoding="utf-8") as handle:
        return json.load(handle)


def atomic_json(path: Path, payload) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    with temp.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, ensure_ascii=False, indent=2)
    temp.replace(path)


def make_args(exp: dict):
    values = [
        "--model", "lgbm",
        "--configs", exp["config"],
        "--cv", CV,
        "--n-splits", str(N_SPLITS),
        "--folds", str(fold_path),
        "--seed", str(SEED),
        "--tag", TAG,
        "--device", DEVICE,
        "--latent-components", str(exp["nmf"]),
        "--tfidf-topk", str(exp["tfidf"]),
        "--parsed-topk", "1000",
        "--no-submission",
    ]
    if THREADS is not None:
        values += ["--threads", str(THREADS)]
    args = tg.build_parser().parse_args(values)
    args.device = {"gpu": True, "cpu": False, "auto": "auto"}[args.device]
    args.override = tg._parse_override(args.overrides)
    args.gpu_ram_part = tg.resolve_gpu_ram_part(args.gpu_ram_part)
    tg.resolve_model_params(args)
    return args


display(pd.DataFrame(EXPERIMENTS))
print("loaded blocks:", sorted(all_needed))

## 6. 8개 실험 실행

각 실험의 OOF/test/log가 Drive에 저장된 뒤 완료 상태를 기록한다. 셀이 중단되면 다시
실행해 완료 항목을 건너뛴다.

In [ ]:
def marker_complete(marker: dict) -> bool:
    return all(Path(marker.get(name, "")).is_file() for name in ("oof_path", "test_path", "log_path"))


state = load_json(STATE_PATH, {})
failures = load_json(FAILURE_PATH, {})
suite_started = time.perf_counter()

for index, exp in enumerate(EXPERIMENTS, start=1):
    key = exp["key"]
    if RESUME_COMPLETED and key in state and marker_complete(state[key]):
        print(f"[{index}/{len(EXPERIMENTS)}] SKIP {key}")
        continue

    print(f"\n[{index}/{len(EXPERIMENTS)}] START {key}: {exp['desc']}")
    args = make_args(exp)
    try:
        result = tg.run_config(data, config=exp["config"], cv=CV, args=args)
        stem = result["stem"]
        marker = {
            **exp,
            "stem": stem,
            "oof_macro_f1": result["oof_macro_f1"],
            "oof_macro_f1_singleton": result["oof_macro_f1_singleton"],
            "oof_accuracy": result["oof_accuracy"],
            "n_features_per_fold": result["n_features_per_fold"],
            "fold_macro_f1": result["fold_macro_f1"],
            "elapsed_seconds": result["elapsed_seconds"],
            "model_params": result["model_params"],
            "oof_path": str(tg.ARTIFACTS / "oof" / f"oof_{stem}.csv"),
            "test_path": str(tg.ARTIFACTS / "test_predictions" / f"test_{stem}.csv"),
            "log_path": str(tg.ARTIFACTS / "logs" / f"{stem}.json"),
            "completed_at_utc": datetime.now(timezone.utc).isoformat(),
        }
        if not marker_complete(marker):
            raise RuntimeError(f"학습 후 산출물 누락: {marker}")
        state[key] = marker
        failures.pop(key, None)
        atomic_json(STATE_PATH, state)
        atomic_json(FAILURE_PATH, failures)
        print(f"[{index}/{len(EXPERIMENTS)}] DONE {key}: {result['oof_macro_f1']:.6f}")
    except Exception as exc:
        failures[key] = {
            **exp,
            "type": type(exc).__name__,
            "message": str(exc),
            "traceback": traceback.format_exc(),
            "failed_at_utc": datetime.now(timezone.utc).isoformat(),
        }
        atomic_json(FAILURE_PATH, failures)
        print(f"[{index}/{len(EXPERIMENTS)}] FAILED {key}: {type(exc).__name__}: {exc}")

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "git_branch": branch_name,
    "git_commit": commit_sha,
    "python": platform.python_version(),
    "lightgbm": lightgbm.__version__,
    "scikit_learn": sklearn.__version__,
    "cv": CV,
    "fold_column": "fold_group5",
    "n_splits": N_SPLITS,
    "seed": SEED,
    "class_order": list(observed_classes),
    "pair_rule_applied": False,
    "experiments": EXPERIMENTS,
}
atomic_json(OUTPUT_ROOT / "manifest.json", manifest)
print(f"elapsed: {(time.perf_counter() - suite_started) / 60:.1f} min")
print(f"completed: {len(state)}/{len(EXPERIMENTS)}, failed: {len(failures)}")

## 7. 단일 OOF 비교표

이 표는 Pair rule 적용 전 raw OOF만 사용한다.

In [ ]:
state = load_json(STATE_PATH, {})
rows = []
base_f1 = state.get("base", {}).get("oof_macro_f1")
for exp in EXPERIMENTS:
    marker = state.get(exp["key"])
    if marker is None:
        continue
    rows.append(
        {
            "experiment": exp["key"],
            "description": exp["desc"],
            "config": exp["config"],
            "nmf_components": exp["nmf"],
            "tfidf_topk": exp["tfidf"],
            "oof_macro_f1": marker["oof_macro_f1"],
            "delta_vs_lgbm_base": (
                marker["oof_macro_f1"] - base_f1 if base_f1 is not None else np.nan
            ),
            "singleton_f1": marker["oof_macro_f1_singleton"],
            "fold_std": float(np.std(marker["fold_macro_f1"])),
            "features_mean": float(np.mean(marker["n_features_per_fold"])),
            "elapsed_min": marker["elapsed_seconds"] / 60,
        }
    )

single_summary = pd.DataFrame(rows).sort_values("oof_macro_f1", ascending=False)
single_summary.to_csv(OUTPUT_ROOT / "single_oof_summary.csv", index=False)
display(single_summary.style.format({
    "oof_macro_f1": "{:.6f}",
    "delta_vs_lgbm_base": "{:+.6f}",
    "singleton_f1": "{:.6f}",
    "fold_std": "{:.6f}",
    "features_mean": "{:.0f}",
    "elapsed_min": "{:.1f}",
}))
print("saved:", OUTPUT_ROOT / "single_oof_summary.csv")

## 8. v002 raw OOF와 50:50 평균

`V002_OOF`가 비어 있으면 건너뛴다. v002와 새 후보 모두 동일한 canonical Group5 OOF여야
한다. 클래스 순서, ID, `y_true`를 확인한 뒤 확률을 50:50 평균한다. Pair rule과 로짓
보정은 적용하지 않는다.

In [ ]:
from sklearn.metrics import f1_score


def load_oof(path: str | Path) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    frame = pd.read_csv(path)
    classes = list(observed_classes)
    if "ID" in frame.columns:
        id_col, y_col = "ID", "y_true"
        prob_cols = [f"p_{name}" for name in classes]
    elif "sample_id" in frame.columns:
        id_col, y_col = "sample_id", "true_label"
        prob_cols = [f"prob_class_{i}" for i in range(len(classes))]
    else:
        raise ValueError(f"지원하지 않는 OOF schema: {list(frame.columns)[:8]}")
    missing = [column for column in [id_col, y_col, *prob_cols] if column not in frame.columns]
    if missing:
        raise ValueError(f"OOF에 없는 열: {missing[:8]}")
    if frame[id_col].astype(str).duplicated().any():
        raise ValueError("OOF에 중복 ID가 있습니다")
    ids = frame[id_col].astype(str).to_numpy()
    y_true = frame[y_col].astype(str).to_numpy()
    proba = frame[prob_cols].to_numpy(np.float64)
    if not np.isfinite(proba).all() or (proba < 0).any():
        raise ValueError("OOF 확률에 NaN/음수/무한대가 있습니다")
    totals = proba.sum(axis=1, keepdims=True)
    if (totals <= 0).any():
        raise ValueError("합이 0인 OOF 확률 행이 있습니다")
    return ids, y_true, proba / totals


def macro_f1(y_true: np.ndarray, proba: np.ndarray) -> float:
    pred = np.asarray(observed_classes)[proba.argmax(axis=1)]
    return float(
        f1_score(
            y_true, pred, labels=list(observed_classes),
            average="macro", zero_division=0,
        )
    )


if not V002_OOF:
    print("V002_OOF가 비어 있어 50:50 평가는 건너뜁니다.")
    print("Pair rule 적용 전 v002 raw Group5 OOF 경로를 설정하고 이 셀만 다시 실행하세요.")
else:
    baseline_ids, baseline_y, baseline_proba = load_oof(V002_OOF)
    baseline_score = macro_f1(baseline_y, baseline_proba)
    blend_rows = []
    for exp in EXPERIMENTS:
        marker = state.get(exp["key"])
        if marker is None:
            continue
        candidate_ids, candidate_y, candidate_proba = load_oof(marker["oof_path"])
        candidate_lookup = {sample_id: i for i, sample_id in enumerate(candidate_ids)}
        if set(candidate_lookup) != set(baseline_ids.tolist()):
            raise ValueError(f"{exp['key']}: v002와 OOF ID 집합이 다릅니다")
        order = np.asarray([candidate_lookup[sample_id] for sample_id in baseline_ids])
        candidate_y = candidate_y[order]
        candidate_proba = candidate_proba[order]
        if not np.array_equal(candidate_y, baseline_y):
            raise ValueError(f"{exp['key']}: v002와 y_true가 다릅니다")
        candidate_score = macro_f1(baseline_y, candidate_proba)
        blended = 0.5 * baseline_proba + 0.5 * candidate_proba
        blend_score = macro_f1(baseline_y, blended)
        blend_rows.append(
            {
                "experiment": exp["key"],
                "v002_oof_macro_f1": baseline_score,
                "candidate_oof_macro_f1": candidate_score,
                "blend_50_50_oof_macro_f1": blend_score,
                "increment_vs_v002": blend_score - baseline_score,
            }
        )
    blend_summary = pd.DataFrame(blend_rows).sort_values(
        "increment_vs_v002", ascending=False
    )
    blend_summary.to_csv(OUTPUT_ROOT / "v002_50_50_oof_summary.csv", index=False)
    display(blend_summary.style.format({
        "v002_oof_macro_f1": "{:.6f}",
        "candidate_oof_macro_f1": "{:.6f}",
        "blend_50_50_oof_macro_f1": "{:.6f}",
        "increment_vs_v002": "{:+.6f}",
    }))
    print("saved:", OUTPUT_ROOT / "v002_50_50_oof_summary.csv")

## 9. 결과 묶기

Drive에는 원본 결과가 계속 남는다. 아래 셀은 공유·다운로드용 zip을 추가로 만든다.

In [ ]:
archive_base = WORK_ROOT / f"{TAG}_{commit_sha[:8]}"
archive_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=OUTPUT_ROOT))
print("archive:", archive_path)
print("size MB:", round(archive_path.stat().st_size / 1024**2, 2))
print("Pair rule 적용: False")
print("DACON 업로드: 수행하지 않음")

if IN_COLAB:
    from google.colab import files

    print(f"필요하면 실행: files.download({str(archive_path)!r})")